In [ ]:
import os
import copy
import time

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
import torchvision

# Import only the FP32 ResNet. `from models import *` would also bring in models/quant_layer.weight_quantize_fn
# (which normalizes the weights and divides them in place), shadowing the Code8 version defined below.
from models import resnet20_cifar

# On Google Colab: mount Drive and chdir into the `software` folder first, e.g.
# from google.colab import drive; drive.mount('/content/drive'); os.chdir('/content/drive/MyDrive/.../software')

ckpt_path = 'result/resnet20/model_best.pth.tar'
assert os.path.exists(ckpt_path), 'run this notebook from software/ (cannot find {})'.format(ckpt_path)

if torch.cuda.is_available():
    device = torch.device('cuda')
    cudnn.benchmark = True       # fixed 32x32 input -> let cuDNN pick the fastest conv kernels
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print('device:', device)

## Data (preloaded onto the device)

In [ ]:
# Same normalization as HW3
mean = torch.tensor([0.491, 0.482, 0.447]).view(1, 3, 1, 1)
std  = torch.tensor([0.247, 0.243, 0.262]).view(1, 3, 1, 1)

def to_tensor(dataset, idx=None):
    # Same result as transforms.ToTensor() + Normalize, but done once for the whole set:
    # uint8 NHWC -> float NCHW in [0, 1] -> normalized
    data, targets = dataset.data, np.array(dataset.targets)
    if idx is not None:
        data, targets = data[idx], targets[idx]
    x = torch.from_numpy(data).permute(0, 3, 1, 2).contiguous().float().div(255)
    x = (x - mean) / std
    return x.to(device), torch.from_numpy(targets).long().to(device)

test_dataset  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True)
train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True)

x_test, y_test = to_tensor(test_dataset)

# Calibration set for picking alpha by accuracy: 5,000 *training* images (no augmentation),
# so the test set is only used for the reported numbers.
calib_idx = np.random.RandomState(0).permutation(len(train_dataset))[:5000]
x_calib, y_calib = to_tensor(train_dataset, calib_idx)

print('test :', tuple(x_test.shape))
print('calib:', tuple(x_calib.shape))


@torch.inference_mode()
def evaluate(model, x, y, batch_size=1000):
    # Top-1 accuracy (%) on preloaded tensors
    model.eval()
    correct = 0
    for i in range(0, len(x), batch_size):
        pred = model(x[i:i + batch_size]).argmax(dim=1)
        correct += (pred == y[i:i + batch_size]).sum().item()
    return 100.0 * correct / len(x)

## Load the trained FP32 ResNet-20 from HW3

In [ ]:
model = resnet20_cifar().to(device)

checkpoint = torch.load(ckpt_path, map_location=device)
model.load_state_dict(checkpoint['state_dict'])
model.eval()

fp32_state = copy.deepcopy(model.state_dict())   # untouched FP32 weights, restored before every experiment
conv_names = [name for name, m in model.named_modules() if isinstance(m, nn.Conv2d)]   # same order as model.modules()

fp32_acc = evaluate(model, x_test, y_test)
fp32_calib_acc = evaluate(model, x_calib, y_calib)
print('checkpoint: epoch {}, best_prec {:.3f}%'.format(checkpoint['epoch'], checkpoint['best_prec']))
print('FP32 test accuracy       : {:.2f}%'.format(fp32_acc))
print('FP32 calibration accuracy: {:.2f}%'.format(fp32_calib_acc))
print('{} conv layers: {}'.format(len(conv_names), conv_names))

## Weight quantizer from [Code8]

`w_bit` includes the sign bit: the magnitude gets `b = w_bit - 1` bits, so the integer levels are
−(2^b − 1) … +(2^b − 1) and the step is Δ = α / (2^b − 1).

| w_bit | integer range | levels | step Δ |
|---|---|---|---|
| 8 | −127 … 127 | 255 | α / 127 |
| 4 | −7 … 7 | 15 | α / 7 |

In [ ]:
# weight_quantization / weight_quantize_fn from [Code8]. The quantization logic is unchanged; forward is made a
# @staticmethod and the Function is applied through the class (uq.apply), as current PyTorch expects.
# PTQ needs no gradients, so there is no backward.

def weight_quantization(b):

    def uniform_quant(x, b):
        xdiv = x.mul((2 ** b - 1))
        xhard = xdiv.round().div(2 ** b - 1)
        return xhard

    class uq(torch.autograd.Function):

        @staticmethod
        def forward(ctx, input, alpha):
            input_d = input/alpha                          # weights are first divided by alpha
            input_c = input_d.clamp(min=-1, max=1)       # then clipped to [-1,1]
            sign = input_c.sign()
            input_abs = input_c.abs()
            input_q = uniform_quant(input_abs, b).mul(sign)
            ctx.save_for_backward(input, input_q)
            input_q_out = input_q.mul(alpha)               # rescale to the original range
            return input_q_out

    return uq.apply


class weight_quantize_fn(nn.Module):
    def __init__(self, w_bit):
        super(weight_quantize_fn, self).__init__()
        self.w_bit = w_bit-1
        self.weight_q = weight_quantization(b=self.w_bit)
        self.wgt_alpha = 0.0

    def forward(self, weight):
        weight_q = self.weight_q(weight, self.wgt_alpha)

        return weight_q


def quantize(w, alpha, w_bit):
    # alpha: a scalar, or a tensor that broadcasts against w (used to try many alphas at once)
    weight_quant = weight_quantize_fn(w_bit=w_bit)
    weight_quant.wgt_alpha = torch.as_tensor(alpha, dtype=w.dtype, device=w.device)
    return weight_quant(w)

## Choosing α

α trades off two kinds of error:

* **Rounding error** — every weight inside [−α, α] is rounded to the nearest level; step Δ = α / (2^(b−1) − 1), error ≤ Δ/2.
  Larger α → coarser step for *all* weights.
* **Clipping error** — weights with |w| > α are clamped to ±α. Smaller α → more (large) weights distorted.

Three strategies, each giving one α per conv layer:

1. **max** — α = max|W|: no clipping at all, but the coarsest step.
2. **mse** — α = argmin ‖W − Q(W; α)‖², searched over 200 candidates in (0, max|W|]. Uses only the weights (no data).
3. **calib** — α = r · max|W| with one global ratio r for all layers, picked by accuracy on the 5k-image calibration set.

In [ ]:
def alpha_max(w, w_bit):
    return w.abs().max().item()


def alpha_mse(w, w_bit, n_grid=200):
    # Try all candidates at once: alphas [n_grid, 1] broadcast against the flattened weights [1, N]
    w_flat = w.reshape(1, -1)
    cands = torch.linspace(0.005, 1.0, n_grid, device=w.device) * w_flat.abs().max()
    err = (quantize(w_flat, cands[:, None], w_bit) - w_flat).pow(2).mean(dim=1)
    return cands[err.argmin()].item()


def alpha_ratio(r):
    # alpha = r * max|W|, with one ratio r shared by all layers
    return lambda w, w_bit: r * w.abs().max().item()


def quantize_model(model, w_bit, choose_alpha):
    # Restore the FP32 weights, then quantize the weight of every Conv2d layer in place.
    # Returns the alpha used for each conv layer (model.modules() order, same as conv_names).
    model.load_state_dict(fp32_state)
    alphas = []
    with torch.no_grad():
        for layer in model.modules():
            if isinstance(layer, torch.nn.Conv2d):
                alpha = choose_alpha(layer.weight, w_bit)
                layer.weight.copy_(quantize(layer.weight, alpha, w_bit))
                alphas.append(alpha)
    return alphas


# Sanity check: every conv weight is now an integer multiple of alpha/(2^(b-1)-1) within the allowed range
for w_bit in (8, 4):
    alphas = quantize_model(model, w_bit, alpha_max)
    n_lv = 2 ** (w_bit - 1) - 1
    convs = [m for m in model.modules() if isinstance(m, nn.Conv2d)]
    w_int = [(m.weight / (a / n_lv)).round().long() for m, a in zip(convs, alphas)]
    n_distinct = [torch.unique(w).numel() for w in w_int]
    assert max(w.abs().max().item() for w in w_int) <= n_lv
    print('{}-bit: integer range [-{}, {}], distinct values per layer: {}'.format(w_bit, n_lv, n_lv, n_distinct))

print('\nlayer1[0].conv1, 4-bit integer weights of filter 0, input channel 0:')
print(w_int[1][0, 0])

## Quantize, evaluate, report

In [ ]:
results = {'FP32': fp32_acc}
alphas_used = {}
calib_acc = {}
ratios = np.round(np.arange(0.20, 1.001, 0.05), 2)

t0 = time.time()
for w_bit in (8, 4):
    # 1) and 2): per-layer alpha from the weights alone
    for name, choose_alpha in [('max', alpha_max), ('mse', alpha_mse)]:
        alphas_used[(w_bit, name)] = quantize_model(model, w_bit, choose_alpha)
        results['{}-bit, alpha={}'.format(w_bit, name)] = evaluate(model, x_test, y_test)

    # 3) global ratio r picked on the calibration set
    calib_acc[w_bit] = []
    for r in ratios:
        quantize_model(model, w_bit, alpha_ratio(r))
        calib_acc[w_bit].append(evaluate(model, x_calib, y_calib))
    best_r = float(ratios[int(np.argmax(calib_acc[w_bit]))])
    alphas_used[(w_bit, 'calib')] = quantize_model(model, w_bit, alpha_ratio(best_r))
    results['{}-bit, alpha=calib (r={:.2f})'.format(w_bit, best_r)] = evaluate(model, x_test, y_test)
print('done in {:.1f}s'.format(time.time() - t0))

print('\n{:<32s}{:>10s}{:>10s}'.format('model', 'test acc', 'drop'))
for k, acc in results.items():
    print('{:<32s}{:>9.2f}%{:>9.2f}%'.format(k, acc, fp32_acc - acc))

In [ ]:
# Calibration accuracy vs. clipping ratio. Dotted lines: mean alpha/max|W| chosen by the MSE criterion.
w_max = np.array([fp32_state[n + '.weight'].abs().max().item() for n in conv_names])

fig, ax = plt.subplots(figsize=(7, 4))
for w_bit, color in [(8, 'tab:blue'), (4, 'tab:orange')]:
    ax.plot(ratios, calib_acc[w_bit], 'o-', color=color, label='{}-bit'.format(w_bit))
    ax.axvline(np.mean(np.array(alphas_used[(w_bit, 'mse')]) / w_max), color=color, ls=':',
               label='{}-bit, mean MSE-optimal ratio'.format(w_bit))
ax.axhline(fp32_calib_acc, color='gray', ls='--', label='FP32')
ax.set_xlabel('clipping ratio  r = alpha / max|W|')
ax.set_ylabel('calibration accuracy (%)')
ax.set_title('ResNet-20 weight PTQ: accuracy vs. clipping value')
ax.legend()
ax.grid(alpha=0.3)
plt.show()